# Phase 3.3 – Segmentation Clients (K-Means Clustering)

## Objectif
Identifier des segments clients homogènes à partir de leurs caractéristiques démographiques et de revenus, afin de cibler les actions marketing de manière plus précise.

## Source de données
- `ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS`

## Approche
- Algorithme : **K-Means Clustering**
- Features : âge, revenu annuel
- Évaluation : méthode du coude (Elbow Method) + Silhouette Score

In [ ]:
# Installation des dépendances (si nécessaire)
# !pip install pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print('Librairies chargées avec succès ✅')

## 1. Chargement des données

In [ ]:
# Charger le fichier CSV exporté depuis Snowflake :
# ANYCOMPANY_LAB.ANALYTICS.CUSTOMER_SEGMENTS
df = pd.read_csv('customer_segments.csv')

df.columns = df.columns.str.upper()
df = df[df['AGE'].notna() & df['ANNUAL_INCOME'].notna()].reset_index(drop=True)

print(f'Données chargées : {df.shape[0]} clients, {df.shape[1]} colonnes')
df.head(10)

In [ ]:
# Statistiques descriptives
df[['AGE', 'ANNUAL_INCOME']].describe()

## 2. Préparation des features

In [ ]:
# Sélection des features pour le clustering
features = ['AGE', 'ANNUAL_INCOME']
X = df[features].copy()

# Normalisation (StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features normalisées ✅')
print(f'Moyenne après scaling : {X_scaled.mean(axis=0).round(4)}')
print(f'Écart-type après scaling : {X_scaled.std(axis=0).round(4)}')

## 3. Choix du nombre de clusters – Méthode du coude

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
    score = silhouette_score(X_scaled, kmeans.labels_)
    silhouette_scores.append(score)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Nombre de clusters (K)')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode du Coude (Elbow Method)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'rs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Nombre de clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score par K')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f'\nMeilleur K selon Silhouette Score : {best_k} (score = {max(silhouette_scores):.4f})')

## 4. Modèle K-Means final

In [ ]:
# Entraînement avec le meilleur K
K_OPTIMAL = best_k
kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df['CLUSTER'] = kmeans_final.fit_predict(X_scaled)

print(f'Modèle K-Means entraîné avec K={K_OPTIMAL} ✅')
print(f'Silhouette Score final : {silhouette_score(X_scaled, df["CLUSTER"]):.4f}')
print(f'\nRépartition des clients par cluster :')
print(df['CLUSTER'].value_counts().sort_index())

## 5. Visualisation des clusters

In [ ]:
colors = plt.cm.Set1(np.linspace(0, 1, K_OPTIMAL))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot AGE vs ANNUAL_INCOME
for cluster_id in range(K_OPTIMAL):
    mask = df['CLUSTER'] == cluster_id
    axes[0].scatter(
        df.loc[mask, 'AGE'],
        df.loc[mask, 'ANNUAL_INCOME'],
        c=[colors[cluster_id]], label=f'Cluster {cluster_id}',
        alpha=0.6, s=20
    )

# Centroïdes
centroids_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
axes[0].scatter(centroids_original[:, 0], centroids_original[:, 1],
                c='black', marker='X', s=200, zorder=5, label='Centroïdes')
axes[0].set_xlabel('Âge')
axes[0].set_ylabel('Revenu annuel ($)')
axes[0].set_title('Segmentation Clients – K-Means')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution des clusters
cluster_counts = df['CLUSTER'].value_counts().sort_index()
axes[1].bar([f'Cluster {i}' for i in cluster_counts.index],
            cluster_counts.values,
            color=colors[:K_OPTIMAL])
axes[1].set_title('Nombre de clients par cluster')
axes[1].set_ylabel('Nombre de clients')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('clusters_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Profil des segments

In [ ]:
# Profil moyen de chaque cluster
cluster_profile = df.groupby('CLUSTER').agg(
    NB_CLIENTS     = ('CUSTOMER_ID', 'count'),
    AGE_MOYEN      = ('AGE', 'mean'),
    REVENU_MOYEN   = ('ANNUAL_INCOME', 'mean'),
    AGE_MEDIAN     = ('AGE', 'median'),
    REVENU_MEDIAN  = ('ANNUAL_INCOME', 'median')
).round(1)

cluster_profile['PCT_CLIENTS'] = (cluster_profile['NB_CLIENTS'] / len(df) * 100).round(1)
cluster_profile['REVENU_MOYEN'] = cluster_profile['REVENU_MOYEN'].apply(lambda x: f'${x:,.0f}')
cluster_profile['REVENU_MEDIAN'] = cluster_profile['REVENU_MEDIAN'].apply(lambda x: f'${x:,.0f}')

print('=== PROFIL DES SEGMENTS CLIENTS ===')
print(cluster_profile.to_string())

# Distribution du revenu par cluster
plt.figure(figsize=(12, 5))
for cluster_id in range(K_OPTIMAL):
    mask = df['CLUSTER'] == cluster_id
    plt.hist(df.loc[mask, 'ANNUAL_INCOME'], bins=30, alpha=0.6,
             label=f'Cluster {cluster_id}', color=colors[cluster_id])
plt.xlabel('Revenu annuel ($)')
plt.ylabel('Nombre de clients')
plt.title('Distribution du revenu par cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('revenue_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribution géographique par cluster
geo_cluster = df.groupby(['REGION', 'CLUSTER']).size().unstack(fill_value=0)

plt.figure(figsize=(12, 6))
geo_cluster.plot(kind='bar', stacked=True, figsize=(12, 6),
                 color=colors[:K_OPTIMAL])
plt.title('Répartition des clusters par région')
plt.xlabel('Région')
plt.ylabel('Nombre de clients')
plt.xticks(rotation=45, ha='right')
plt.legend([f'Cluster {i}' for i in range(K_OPTIMAL)], loc='upper right')
plt.tight_layout()
plt.savefig('geo_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Interprétation et recommandations marketing

In [ ]:
# Nommage des segments selon leur profil
segment_labels = {}
for cluster_id in range(K_OPTIMAL):
    mask = df['CLUSTER'] == cluster_id
    age_moy = df.loc[mask, 'AGE'].mean()
    revenu_moy = df.loc[mask, 'ANNUAL_INCOME'].mean()

    if age_moy < 35 and revenu_moy < 60000:
        label = 'Jeunes à faible revenu'
    elif age_moy < 35 and revenu_moy >= 60000:
        label = 'Jeunes professionnels aisés'
    elif age_moy >= 35 and age_moy < 55 and revenu_moy >= 100000:
        label = 'Actifs à fort pouvoir d\'achat'
    elif age_moy >= 35 and age_moy < 55:
        label = 'Actifs classe moyenne'
    else:
        label = 'Seniors'

    segment_labels[cluster_id] = label

df['SEGMENT_LABEL'] = df['CLUSTER'].map(segment_labels)

print('=== SEGMENTS IDENTIFIÉS ET RECOMMANDATIONS MARKETING ===')
for cluster_id, label in segment_labels.items():
    mask = df['CLUSTER'] == cluster_id
    nb = mask.sum()
    pct = nb / len(df) * 100
    age = df.loc[mask, 'AGE'].mean()
    revenu = df.loc[mask, 'ANNUAL_INCOME'].mean()
    print(f'\nCluster {cluster_id} – {label}')
    print(f'  Taille : {nb} clients ({pct:.1f}%)')
    print(f'  Âge moyen : {age:.1f} ans | Revenu moyen : ${revenu:,.0f}')

print('\n✅ Segmentation terminée')

In [ ]:
print('✅ Notebook customer_segmentation terminé avec succès.')